# Colab bootstrap — private-repo escape hatch

Use this notebook when the Colab GitHub integration can't see your private repo.
It clones the repo using a PAT from Colab Secrets, then hands off to either
`colab_phase0_dml.ipynb` or `colab_phase1_tradefm.ipynb`.

**One-time setup:**
1. On GitHub: *Settings → Developer settings → Personal access tokens → Tokens (classic)
   → Generate new token (classic)*. Scope = `repo`. Expiry 30d.
2. In Colab: left-side **🔑 key icon** → *+ Add new secret*.
   Name: **`GITHUB_TOKEN`**. Value: the PAT. Toggle *Notebook access* ON.
3. *Runtime → Change runtime type → A100* (for Phase 1) or T4 (for Phase 0).

**Then just run the one cell below.**

In [ ]:
import os
from google.colab import userdata, drive

GH_USER = 'nahomar'
REPO_NAME = 'market-pattern-bot'

try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError(
        'Add GITHUB_TOKEN in Colab Secrets (🔑 icon → + Add new secret).'
    ) from e
if not TOKEN:
    raise RuntimeError('GITHUB_TOKEN is empty — refresh and retry.')

AUTH_URL  = f'https://{TOKEN}@github.com/{GH_USER}/{REPO_NAME}.git'
CLEAN_URL = f'https://github.com/{GH_USER}/{REPO_NAME}.git'
CLONE_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(CLONE_DIR):
    !git clone --depth=1 $AUTH_URL $CLONE_DIR
os.chdir(CLONE_DIR)
# Scrub the token out of .git/config so it never lands in Drive
!git remote set-url origin $CLEAN_URL
del TOKEN, AUTH_URL

drive.mount('/content/drive', force_remount=False)
!mkdir -p /content/drive/MyDrive/tradefm_ckpts

!pip install -q -r requirements.txt
!pip install -q numba pyarrow matplotlib wandb

print('\n✅ repo ready at', CLONE_DIR)
print('\nNEXT STEP — open the left-side 📁 file browser:')
print('  /content/market-pattern-bot/notebooks/colab_phase0_dml.ipynb')
print('  /content/market-pattern-bot/notebooks/colab_phase1_tradefm.ipynb')
print('\nRight-click → "Open notebook" to launch either in a new tab.')

## Optional: open the Phase 0 or Phase 1 notebook inline

If you want to keep everything in one tab, you can just `%run` each phase
as a Python script. The notebook cells below work, but they're less convenient
than opening the actual .ipynb files from the left-side browser (you lose the
per-cell output structure).

```python
# Phase 0 equivalent (not recommended — use the .ipynb instead):
!python odte_smoke.py --device cuda --dml-steps 2000

# Phase 1 equivalent:
!python odte_phase1_smoke.py --device cuda --days 10 --steps 5000
```